In [1]:
#Sptial attention

import torch
import os
import torch.nn as nn
from torchvision import datasets
from torchvision import transforms
from torchvision.models import vit_b_16
from torch.utils.data import DataLoader,random_split

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [3]:
from torch.cuda import manual_seed
from google.colab import drive
drive.mount('/content/drive')

path = '/content/drive/MyDrive/Breast_Cancer/dataset_cancer_v1/classificacao_binaria/40X'


transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor()
])

full_dataset = datasets.ImageFolder(path,transform = transform)

train_size = int(0.8*len(full_dataset))
val_size = int(0.1*len(full_dataset))
test_size = len(full_dataset) - train_size - val_size


train_dataset,val_dataset,test_dataset = random_split(
    full_dataset,
    [train_size,val_size,test_size],
    generator= torch.Generator().manual_seed(42)
)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
#Data Loader

train_loader = DataLoader(
    train_dataset,
    batch_size= 32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [5]:
class SpatialAttention(nn.Module):
  def __init__(self):
    super().__init__()

    self.conv = nn.Conv2d(
        in_channels=2,
        out_channels=1,
        kernel_size=7,
        padding=3,
        bias=False
    )

    self.sigmoid = nn.Sigmoid()

  def forward(self,x):

    avg_out = torch.mean(
        x,
        dim=1,
        keepdim=True
    )

    max_out,_ = torch.max(
        x,
        dim=1,
        keepdim=True
    )

    attention = torch.cat([avg_out,max_out],dim=1)

    attention =  self.conv(attention)
    attention = self.sigmoid(attention)

    return x*attention

In [6]:
class ViTWithSpatialAttention(nn.Module):
  def __init__(self,num_classes=2):
    super().__init__()

    self.vit = vit_b_16(weights="IMAGENET1K_V1")
    in_features = self.vit.heads.head.in_features

    self.vit.heads = nn.Linear(in_features,512)
    self.attention = SpatialAttention()
    self.classifier = nn.Linear(512,num_classes)

  def forward(self,x):

    x = self.vit(x)
    x=x.unsqueeze(-1).unsqueeze(-1)
    x = self.attention(x)
    x=x.squeeze(-1).squeeze(-1)
    x = self.classifier(x)
    return x


model = ViTWithSpatialAttention().to(device)




In [7]:
import torch.optim

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(),lr=0.0001)

In [8]:
epochs = 10
best_loss = float("inf")

for epoch in range(epochs):
    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    training_loss = running_loss / len(train_loader)
    train_acc = correct / total * 100

    # ---------------- VALIDATION ----------------
    model.eval()
    val_loss = 0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

    validation_loss = val_loss / len(val_loader)
    val_acc = correct / total * 100

    print(f"Epoch {epoch+1}/{epochs}, "
          f"Train Loss: {training_loss:.4f}, Train Acc: {train_acc:.2f}%, "
          f"Val Loss: {validation_loss:.4f}, Val Acc: {val_acc:.2f}%")

    if validation_loss < best_loss:
        best_loss = validation_loss
        torch.save(model.state_dict(), "best_model.pth")

Epoch 1/10, Train Loss: 0.4541, Train Acc: 78.82%, Val Loss: 0.3302, Val Acc: 82.91%
Epoch 2/10, Train Loss: 0.2811, Train Acc: 88.53%, Val Loss: 0.1486, Val Acc: 94.97%
Epoch 3/10, Train Loss: 0.1458, Train Acc: 94.30%, Val Loss: 0.3549, Val Acc: 86.43%
Epoch 4/10, Train Loss: 0.0869, Train Acc: 97.24%, Val Loss: 0.1962, Val Acc: 94.47%
Epoch 5/10, Train Loss: 0.0834, Train Acc: 96.87%, Val Loss: 0.1115, Val Acc: 95.98%
Epoch 6/10, Train Loss: 0.0338, Train Acc: 98.93%, Val Loss: 0.2757, Val Acc: 94.97%
Epoch 7/10, Train Loss: 0.0679, Train Acc: 97.87%, Val Loss: 0.0988, Val Acc: 96.98%
Epoch 8/10, Train Loss: 0.0022, Train Acc: 100.00%, Val Loss: 0.1734, Val Acc: 96.98%
Epoch 9/10, Train Loss: 0.0205, Train Acc: 99.25%, Val Loss: 0.1051, Val Acc: 95.98%
Epoch 10/10, Train Loss: 0.0184, Train Acc: 99.31%, Val Loss: 0.5785, Val Acc: 92.46%


In [9]:
model.load_state_dict(torch.load("best_model.pth"))

<All keys matched successfully>

In [11]:
#Accurcy

model.eval()
correct =0
total = 0

with torch.no_grad():

  for images,labels in test_loader:

    images = images.to(device)
    labels = labels.to(device)

    output = model(images)

    _, predicated = torch.max(output,1)

    total+= labels.size(0)

    correct += (predicated == labels).sum().item()

test_accy = 100*correct / total

print(f"Test Accuracy: {test_accy:.4f}")


Test Accuracy: 96.0000
